# Ensemble V30 モデル評価

このノートブックでは、ensemble_v31モデルの評価を行います。
- Stratified Group K-Foldを使用した各foldの再現
- 各foldでのCMI-score評価
- アンサンブル予測の生成
- 最終的な評価結果の分析

In [2]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple, Any
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import tensorflow as tf
from tensorflow import keras

# プロジェクトのルートディレクトリを追加
import sys
sys.path.append('..')

from src.utils.cmi_evaluation import calculate_cmi_score
from src.utils.pipeline import Preprocessor
from src.trainers.multimodal_trainer_v31 import MultimodalTrainerV30

# 日本語フォント設定
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print("✅ ライブラリ読み込み完了")

ImportError: cannot import name 'MultimodalTrainerV30' from 'src.trainers.multimodal_trainer_v31' (/mnt/c/Users/ShunK/works/CMI_comp/notebooks/../src/trainers/multimodal_trainer_v31.py)

## 設定とデータ読み込み

In [ ]:
# 設定
EXPERIMENT_NAME = "20250717_preproc_train_v31"
TRAINER_NAME = "20250717_preproc_train_v31"
N_FOLDS = 5
RANDOM_SEED = 42

# パス設定
BASE_DIR = Path("../output/experiments")
DATA_DIR = BASE_DIR / EXPERIMENT_NAME / "preprocessed"
MODEL_DIR = BASE_DIR / TRAINER_NAME / "models"
RESULT_DIR = BASE_DIR / TRAINER_NAME / "results"

print(f"実験名: {EXPERIMENT_NAME}")
print(f"データディレクトリ: {DATA_DIR}")
print(f"モデルディレクトリ: {MODEL_DIR}")
print(f"結果ディレクトリ: {RESULT_DIR}")

実験名: 20250717_preproc_train_v31
データディレクトリ: ../output/experiments/20250717_preproc_train_v31/preprocessed
モデルディレクトリ: ../output/experiments/20250717_preproc_train_v31/models
結果ディレクトリ: ../output/experiments/20250717_preproc_train_v31/results


In [ ]:
# データ読み込み関数
def load_data() -> Dict[str, np.ndarray]:
    """前処理済みデータを読み込む"""
    print("📊 データ読み込み中...")
    
    def _load(name: str) -> np.ndarray:
        npy = DATA_DIR / f"{name}.npy"
        pkl = DATA_DIR / f"{name}.pkl"
        if npy.exists():
            return np.load(npy)
        if pkl.exists():
            with open(pkl, "rb") as f:
                return pickle.load(f)
        raise FileNotFoundError(f"{name} ファイルが見つかりません")
    
    X_sensor = _load("train_windows")
    X_demo = _load("train_demographics")
    X_tab = _load("train_tabular")
    X_tof = _load("train_tof_windows")
    y = _load("train_labels")
    info = _load("train_info")
    
    # グループ情報の抽出
    if isinstance(info, list) and len(info) > 0 and isinstance(info[0], dict):
        groups = np.array([
            d.get("subject") if d.get("subject") is not None else d.get("sequence_id")
            for d in info
        ])
    else:
        groups = np.asarray(info)
    
    print(f"センサー: {X_sensor.shape}")
    print(f"人口統計: {X_demo.shape}")
    print(f"表形式: {X_tab.shape}")
    print(f"ToF: {X_tof.shape}")
    print(f"ラベル: {y.shape}")
    print(f"グループ数: {len(groups)}")
    print(f"ユニークラベル: {np.unique(y)}")
    
    return {
        "sensor": X_sensor,
        "demographics": X_demo,
        "tabular": X_tab,
        "tof": X_tof,
        "labels": y,
        "groups": groups,
    }

# データ読み込み
data = load_data()

📊 データ読み込み中...
センサー: (8343, 128, 18)
人口統計: (8343, 7)
表形式: (8343, 389)
ToF: (8343, 128, 5, 8, 8)
ラベル: (8343,)
グループ数: 8343
ユニークラベル: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]


## 保存済みモデルの確認

In [ ]:
# 保存済みモデルの確認
print("📁 保存済みモデルの確認:")
if MODEL_DIR.exists():
    model_files = list(MODEL_DIR.glob("*.keras"))
    print(f"見つかったモデル数: {len(model_files)}")
    for model_file in model_files:
        print(f"  - {model_file.name}")
else:
    print("❌ モデルディレクトリが見つかりません")

# 結果ファイルの確認
print("\n📊 結果ファイルの確認:")
if RESULT_DIR.exists():
    result_files = list(RESULT_DIR.glob("*.json"))
    print(f"見つかった結果ファイル数: {len(result_files)}")
    for result_file in result_files:
        print(f"  - {result_file.name}")
else:
    print("❌ 結果ディレクトリが見つかりません")

📁 保存済みモデルの確認:
見つかったモデル数: 6
  - multimodal_model.keras
  - multimodal_model_v31_fold1.keras
  - multimodal_model_v31_fold2.keras
  - multimodal_model_v31_fold3.keras
  - multimodal_model_v31_fold4.keras
  - multimodal_model_v31_fold5.keras

📊 結果ファイルの確認:
見つかった結果ファイル数: 7
  - evaluation_results.json
  - training_history.json
  - training_history_fold1.json
  - training_history_fold2.json
  - training_history_fold3.json
  - training_history_fold4.json
  - training_history_fold5.json


## 各Foldでの評価

In [ ]:
# 各foldでの評価を実行
def evaluate_fold(fold: int, model_path: Path, data: Dict[str, np.ndarray]) -> Dict[str, Any]:
    """単一foldのモデルを評価"""
    print(f"\n🔄 Fold {fold} の評価中...")
    
    # モデル読み込み
    model = keras.models.load_model(model_path)
    print(f"✅ モデル読み込み完了: {model_path.name}")
    
    # 同じfold分割を再現
    skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    splits = list(skf.split(np.arange(len(data["labels"])), data["labels"], data["groups"]))
    train_idx, val_idx = splits[fold - 1]  # foldは1-indexed
    
    # 検証データでの予測
    X_s_val = data["sensor"][val_idx]
    X_d_val = data["demographics"][val_idx]
    X_t_val = data["tabular"][val_idx]
    X_f_val = data["tof"][val_idx]
    y_val = data["labels"][val_idx]
    
    # 予測実行
    preds = model.predict([X_s_val, X_d_val, X_t_val, X_f_val], verbose=0)
    pred_probs = preds
    pred_labels = preds.argmax(axis=1)
    
    # 評価指標計算
    f1_macro = f1_score(y_val, pred_labels, average="macro")
    f1_weighted = f1_score(y_val, pred_labels, average="weighted")
    
    # CMI-score計算
    cmi_score, binary_f1, macro_f1, test_accuracy = calculate_cmi_score(pred_labels, y_val)
    
    # 分類レポート
    report = classification_report(y_val, pred_labels, output_dict=True)
    
    results = {
        "fold": fold,
        "val_indices": val_idx,
        "predictions": pred_labels,
        "prediction_probs": pred_probs,
        "true_labels": y_val,
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "cmi_score": float(cmi_score),
        "binary_f1": float(binary_f1),
        "macro_f1": float(macro_f1),
        "test_accuracy": float(test_accuracy),
        "classification_report": report,
    }
    
    print(f"Fold {fold} 結果:")
    print(f"  F1 Macro: {f1_macro:.4f}")
    print(f"  CMI Score: {cmi_score:.4f}")
    print(f"  Accuracy: {test_accuracy:.4f}")
    
    return results

# 全foldの評価実行
fold_results = []
all_predictions = {}
all_prediction_probs = {}

for fold in range(1, N_FOLDS + 1):
    model_path = MODEL_DIR / f"multimodal_model_v31_fold{fold}.keras"
    
    if model_path.exists():
        results = evaluate_fold(fold, model_path, data)
        fold_results.append(results)
        
        # 予測結果を保存
        for idx, pred in zip(results["val_indices"], results["predictions"]):
            all_predictions[idx] = pred
        for idx, pred_prob in zip(results["val_indices"], results["prediction_probs"]):
            all_prediction_probs[idx] = pred_prob
    else:
        print(f"❌ Fold {fold} のモデルが見つかりません: {model_path}")

print(f"\n✅ {len(fold_results)} foldの評価完了")


🔄 Fold 1 の評価中...
✅ モデル読み込み完了: multimodal_model_v31_fold1.keras
CMI評価指標計算でエラー: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].
Fold 1 結果:
  F1 Macro: 0.5866
  CMI Score: 0.0000
  Accuracy: 0.0000

🔄 Fold 2 の評価中...


Traceback (most recent call last):
  File "/mnt/c/Users/ShunK/works/CMI_comp/notebooks/../src/utils/cmi_evaluation.py", line 105, in calculate_cmi_score
    # マルチクラス分類ではbinaryは使用できないため、microまたはmacroを使用
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 214, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1239, in f1_score
    return fbeta_score(
           ^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 187, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1413, in fbeta_score
    _, _, f, _ = precision_recal

✅ モデル読み込み完了: multimodal_model_v31_fold2.keras
CMI評価指標計算でエラー: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].
Fold 2 結果:
  F1 Macro: 0.5664
  CMI Score: 0.0000
  Accuracy: 0.0000

🔄 Fold 3 の評価中...


Traceback (most recent call last):
  File "/mnt/c/Users/ShunK/works/CMI_comp/notebooks/../src/utils/cmi_evaluation.py", line 105, in calculate_cmi_score
    # マルチクラス分類ではbinaryは使用できないため、microまたはmacroを使用
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 214, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1239, in f1_score
    return fbeta_score(
           ^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 187, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1413, in fbeta_score
    _, _, f, _ = precision_recal

✅ モデル読み込み完了: multimodal_model_v31_fold3.keras
CMI評価指標計算でエラー: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].
Fold 3 結果:
  F1 Macro: 0.5860
  CMI Score: 0.0000
  Accuracy: 0.0000

🔄 Fold 4 の評価中...


Traceback (most recent call last):
  File "/mnt/c/Users/ShunK/works/CMI_comp/notebooks/../src/utils/cmi_evaluation.py", line 105, in calculate_cmi_score
    # マルチクラス分類ではbinaryは使用できないため、microまたはmacroを使用
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 214, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1239, in f1_score
    return fbeta_score(
           ^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 187, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1413, in fbeta_score
    _, _, f, _ = precision_recal

✅ モデル読み込み完了: multimodal_model_v31_fold4.keras
CMI評価指標計算でエラー: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].
Fold 4 結果:
  F1 Macro: 0.5776
  CMI Score: 0.0000
  Accuracy: 0.0000

🔄 Fold 5 の評価中...


Traceback (most recent call last):
  File "/mnt/c/Users/ShunK/works/CMI_comp/notebooks/../src/utils/cmi_evaluation.py", line 105, in calculate_cmi_score
    # マルチクラス分類ではbinaryは使用できないため、microまたはmacroを使用
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 214, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1239, in f1_score
    return fbeta_score(
           ^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 187, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1413, in fbeta_score
    _, _, f, _ = precision_recal

✅ モデル読み込み完了: multimodal_model_v31_fold5.keras
CMI評価指標計算でエラー: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].
Fold 5 結果:
  F1 Macro: 0.5300
  CMI Score: 0.0000
  Accuracy: 0.0000

✅ 5 foldの評価完了


Traceback (most recent call last):
  File "/mnt/c/Users/ShunK/works/CMI_comp/notebooks/../src/utils/cmi_evaluation.py", line 105, in calculate_cmi_score
    # マルチクラス分類ではbinaryは使用できないため、microまたはmacroを使用
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 214, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1239, in f1_score
    return fbeta_score(
           ^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 187, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1413, in fbeta_score
    _, _, f, _ = precision_recal

## アンサンブル予測の生成

In [ ]:
# アンサンブル予測の生成
def generate_ensemble_predictions(all_prediction_probs: Dict[int, np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    """各foldの予測確率を平均してアンサンブル予測を生成"""
    print("\n🎯 アンサンブル予測生成中...")
    
    # 全サンプルのインデックスを取得
    all_indices = sorted(all_prediction_probs.keys())
    
    # 各サンプルについて、全foldの予測確率を平均
    ensemble_probs = []
    ensemble_labels = []
    
    for idx in all_indices:
        # このサンプルの予測確率を収集
        sample_probs = []
        for fold_result in fold_results:
            if idx in fold_result["val_indices"]:
                val_idx_in_fold = np.where(fold_result["val_indices"] == idx)[0][0]
                sample_probs.append(fold_result["prediction_probs"][val_idx_in_fold])
        
        # 平均を計算
        if sample_probs:
            avg_prob = np.mean(sample_probs, axis=0)
            ensemble_probs.append(avg_prob)
            ensemble_labels.append(avg_prob.argmax())
        else:
            print(f"⚠️  サンプル {idx} の予測が見つかりません")
    
    return np.array(ensemble_probs), np.array(ensemble_labels)

# アンサンブル予測生成
ensemble_probs, ensemble_labels = generate_ensemble_predictions(all_prediction_probs)

# アンサンブル予測の評価
all_indices = sorted(all_prediction_probs.keys())
y_true_ensemble = data["labels"][all_indices]

# 評価指標計算
ensemble_f1_macro = f1_score(y_true_ensemble, ensemble_labels, average="macro")
ensemble_f1_weighted = f1_score(y_true_ensemble, ensemble_labels, average="weighted")
ensemble_cmi_score, ensemble_binary_f1, ensemble_macro_f1, ensemble_accuracy = calculate_cmi_score(
    ensemble_labels, y_true_ensemble
)

print(f"\n🎯 アンサンブル結果:")
print(f"  F1 Macro: {ensemble_f1_macro:.4f}")
print(f"  F1 Weighted: {ensemble_f1_weighted:.4f}")
print(f"  CMI Score: {ensemble_cmi_score:.4f}")
print(f"  Binary F1: {ensemble_binary_f1:.4f}")
print(f"  Macro F1: {ensemble_macro_f1:.4f}")
print(f"  Accuracy: {ensemble_accuracy:.4f}")


🎯 アンサンブル予測生成中...
CMI評価指標計算でエラー: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

🎯 アンサンブル結果:
  F1 Macro: 0.5761
  F1 Weighted: 0.5631
  CMI Score: 0.0000
  Binary F1: 0.0000
  Macro F1: 0.0000
  Accuracy: 0.0000


Traceback (most recent call last):
  File "/mnt/c/Users/ShunK/works/CMI_comp/notebooks/../src/utils/cmi_evaluation.py", line 105, in calculate_cmi_score
    # マルチクラス分類ではbinaryは使用できないため、microまたはmacroを使用
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 214, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1239, in f1_score
    return fbeta_score(
           ^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 187, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py", line 1413, in fbeta_score
    _, _, f, _ = precision_recal

## 結果の可視化と分析

In [ ]:
# 結果の可視化
def plot_fold_comparison(fold_results: List[Dict[str, Any]], ensemble_results: Dict[str, float]):
    """各foldとアンサンブルの結果を比較"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Ensemble V30 モデル評価結果', fontsize=16)
    
    # Fold別F1スコア
    fold_numbers = [r["fold"] for r in fold_results]
    f1_scores = [r["f1_macro"] for r in fold_results]
    cmi_scores = [r["cmi_score"] for r in fold_results]
    
    # F1スコア比較
    ax1.bar(fold_numbers, f1_scores, alpha=0.7, color='blue', label='Individual Folds')
    ax1.axhline(y=ensemble_results['f1_macro'], color='red', linestyle='--', 
                label=f'Ensemble: {ensemble_results["f1_macro"]:.4f}', linewidth=2)
    ax1.set_xlabel('Fold')
    ax1.set_ylabel('F1 Macro Score')
    ax1.set_title('F1 Macro Score by Fold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # CMIスコア比較
    ax2.bar(fold_numbers, cmi_scores, alpha=0.7, color='green', label='Individual Folds')
    ax2.axhline(y=ensemble_results['cmi_score'], color='red', linestyle='--', 
                label=f'Ensemble: {ensemble_results["cmi_score"]:.4f}', linewidth=2)
    ax2.set_xlabel('Fold')
    ax2.set_ylabel('CMI Score')
    ax2.set_title('CMI Score by Fold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 統計情報
    mean_f1 = np.mean(f1_scores)
    std_f1 = np.std(f1_scores)
    mean_cmi = np.mean(cmi_scores)
    std_cmi = np.std(cmi_scores)
    
    ax3.text(0.1, 0.8, f'Mean F1: {mean_f1:.4f}', fontsize=12, transform=ax3.transAxes)
    ax3.text(0.1, 0.7, f'Std F1: {std_f1:.4f}', fontsize=12, transform=ax3.transAxes)
    ax3.text(0.1, 0.6, f'Ensemble F1: {ensemble_results["f1_macro"]:.4f}', fontsize=12, transform=ax3.transAxes)
    ax3.text(0.1, 0.5, f'Improvement: {ensemble_results["f1_macro"] - mean_f1:.4f}', fontsize=12, transform=ax3.transAxes)
    ax3.set_title('F1 Score Statistics')
    ax3.axis('off')
    
    ax4.text(0.1, 0.8, f'Mean CMI: {mean_cmi:.4f}', fontsize=12, transform=ax4.transAxes)
    ax4.text(0.1, 0.7, f'Std CMI: {std_cmi:.4f}', fontsize=12, transform=ax4.transAxes)
    ax4.text(0.1, 0.6, f'Ensemble CMI: {ensemble_results["cmi_score"]:.4f}', fontsize=12, transform=ax4.transAxes)
    ax4.text(0.1, 0.5, f'Improvement: {ensemble_results["cmi_score"] - mean_cmi:.4f}', fontsize=12, transform=ax4.transAxes)
    ax4.set_title('CMI Score Statistics')
    ax4.axis('off')
    
    plt.tight_layout()
    plt.show()

# アンサンブル結果を辞書にまとめる
ensemble_results = {
    'f1_macro': ensemble_f1_macro,
    'f1_weighted': ensemble_f1_weighted,
    'cmi_score': ensemble_cmi_score,
    'binary_f1': ensemble_binary_f1,
    'macro_f1': ensemble_macro_f1,
    'accuracy': ensemble_accuracy
}

# 可視化実行
plot_fold_comparison(fold_results, ensemble_results)

In [3]:
# 混同行列の可視化
def plot_confusion_matrices(fold_results: List[Dict[str, Any]], ensemble_labels: np.ndarray, y_true_ensemble: np.ndarray):
    """混同行列を可視化"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Confusion Matrices - Individual Folds vs Ensemble', fontsize=16)
    
    # 各foldの混同行列
    for i, result in enumerate(fold_results[:5]):  # 最大5foldまで表示
        row = i // 3
        col = i % 3
        ax = axes[row, col]
        
        cm = confusion_matrix(result["true_labels"], result["predictions"])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
        ax.set_title(f'Fold {result["fold"]} (F1: {result["f1_macro"]:.3f})')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
    
    # アンサンブルの混同行列
    if len(fold_results) >= 5:
        ax = axes[1, 2]
        cm_ensemble = confusion_matrix(y_true_ensemble, ensemble_labels)
        sns.heatmap(cm_ensemble, annot=True, fmt='d', cmap='Reds', ax=ax)
        ax.set_title(f'Ensemble (F1: {ensemble_f1_macro:.3f})')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
    
    plt.tight_layout()
    plt.show()

# 混同行列可視化
plot_confusion_matrices(fold_results, ensemble_labels, y_true_ensemble)

NameError: name 'fold_results' is not defined

In [ ]:
# 詳細な分類レポート
def print_detailed_reports(fold_results: List[Dict[str, Any]], ensemble_labels: np.ndarray, y_true_ensemble: np.ndarray):
    """詳細な分類レポートを表示"""
    print("\n" + "="*60)
    print("📊 詳細分類レポート")
    print("="*60)
    
    # 各foldのレポート
    for result in fold_results:
        print(f"\n--- Fold {result['fold']} ---")
        print(f"F1 Macro: {result['f1_macro']:.4f}")
        print(f"CMI Score: {result['cmi_score']:.4f}")
        print(f"Accuracy: {result['test_accuracy']:.4f}")
    
    # アンサンブルレポート
    print(f"\n--- Ensemble ---")
    print(f"F1 Macro: {ensemble_f1_macro:.4f}")
    print(f"F1 Weighted: {ensemble_f1_weighted:.4f}")
    print(f"CMI Score: {ensemble_cmi_score:.4f}")
    print(f"Binary F1: {ensemble_binary_f1:.4f}")
    print(f"Macro F1: {ensemble_macro_f1:.4f}")
    print(f"Accuracy: {ensemble_accuracy:.4f}")
    
    # アンサンブルの詳細分類レポート
    print(f"\n--- Ensemble Classification Report ---")
    print(classification_report(y_true_ensemble, ensemble_labels))

# 詳細レポート表示
print_detailed_reports(fold_results, ensemble_labels, y_true_ensemble)


📊 詳細分類レポート

--- Fold 1 ---
F1 Macro: 0.5866
CMI Score: 0.0000
Accuracy: 0.0000

--- Fold 2 ---
F1 Macro: 0.5664
CMI Score: 0.0000
Accuracy: 0.0000

--- Fold 3 ---
F1 Macro: 0.5860
CMI Score: 0.0000
Accuracy: 0.0000

--- Fold 4 ---
F1 Macro: 0.5776
CMI Score: 0.0000
Accuracy: 0.0000

--- Fold 5 ---
F1 Macro: 0.5300
CMI Score: 0.0000
Accuracy: 0.0000

--- Ensemble ---
F1 Macro: 0.5761
F1 Weighted: 0.5631
CMI Score: 0.0000
Binary F1: 0.0000
Macro F1: 0.0000
Accuracy: 0.0000

--- Ensemble Classification Report ---
              precision    recall  f1-score   support

           0       0.63      0.70      0.66       638
           1       0.44      0.44      0.44       642
           2       0.82      0.78      0.80       161
           3       0.37      0.37      0.37       657
           4       0.39      0.32      0.35       641
           5       0.77      0.82      0.80       179
           6       0.57      0.46      0.51       640
           7       0.54      0.63      0.58       

## 結果の保存

In [ ]:
# 結果の保存
def save_evaluation_results(fold_results: List[Dict[str, Any]], ensemble_results: Dict[str, float], 
                           ensemble_labels: np.ndarray, y_true_ensemble: np.ndarray):
    """評価結果を保存"""
    print("\n💾 評価結果を保存中...")
    
    # 保存ディレクトリ作成
    save_dir = Path("../output/experiments/eval_v31")
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # 結果をまとめる
    evaluation_summary = {
        "experiment_name": EXPERIMENT_NAME,
        "trainer_name": TRAINER_NAME,
        "n_folds": N_FOLDS,
        "random_seed": RANDOM_SEED,
        "fold_results": fold_results,
        "ensemble_results": ensemble_results,
        "ensemble_predictions": ensemble_labels.tolist(),
        "true_labels": y_true_ensemble.tolist(),
        "timestamp": pd.Timestamp.now().isoformat()
    }
    
    # JSON形式で保存
    with open(save_dir / "evaluation_results.json", "w", encoding="utf-8") as f:
        json.dump(evaluation_summary, f, ensure_ascii=False, indent=2)
    
    # CSV形式で予測結果を保存
    predictions_df = pd.DataFrame({
        "true_label": y_true_ensemble,
        "ensemble_prediction": ensemble_labels,
        "sample_index": all_indices
    })
    predictions_df.to_csv(save_dir / "ensemble_predictions.csv", index=False)
    
    # 統計サマリーを保存
    summary_stats = {
        "mean_f1_folds": float(np.mean([r["f1_macro"] for r in fold_results])),
        "std_f1_folds": float(np.std([r["f1_macro"] for r in fold_results])),
        "mean_cmi_folds": float(np.mean([r["cmi_score"] for r in fold_results])),
        "std_cmi_folds": float(np.std([r["cmi_score"] for r in fold_results])),
        "ensemble_f1": ensemble_results["f1_macro"],
        "ensemble_cmi": ensemble_results["cmi_score"],
        "f1_improvement": ensemble_results["f1_macro"] - np.mean([r["f1_macro"] for r in fold_results]),
        "cmi_improvement": ensemble_results["cmi_score"] - np.mean([r["cmi_score"] for r in fold_results])
    }
    
    with open(save_dir / "evaluation_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary_stats, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 結果保存完了: {save_dir}")
    print(f"  - evaluation_results.json: 詳細結果")
    print(f"  - ensemble_predictions.csv: 予測結果")
    print(f"  - evaluation_summary.json: 統計サマリー")
    
    return save_dir

# 結果保存
save_dir = save_evaluation_results(fold_results, ensemble_results, ensemble_labels, y_true_ensemble)


💾 評価結果を保存中...


TypeError: Object of type ndarray is not JSON serializable

## 最終サマリー

In [ ]:
# 最終サマリー表示
print("\n" + "="*60)
print("🎯 Ensemble V30 モデル評価 最終サマリー")
print("="*60)

# 各foldの結果
print("\n📊 各Foldの結果:")
for result in fold_results:
    print(f"  Fold {result['fold']:2d}: F1={result['f1_macro']:.4f}, CMI={result['cmi_score']:.4f}")

# 統計情報
f1_scores = [r["f1_macro"] for r in fold_results]
cmi_scores = [r["cmi_score"] for r in fold_results]

print(f"\n📈 統計情報:")
print(f"  F1 Macro - Mean: {np.mean(f1_scores):.4f}, Std: {np.std(f1_scores):.4f}")
print(f"  CMI Score - Mean: {np.mean(cmi_scores):.4f}, Std: {np.std(cmi_scores):.4f}")

# アンサンブル効果
print(f"\n🎯 アンサンブル効果:")
print(f"  F1 Macro: {ensemble_f1_macro:.4f} (改善: {ensemble_f1_macro - np.mean(f1_scores):+.4f})")
print(f"  CMI Score: {ensemble_cmi_score:.4f} (改善: {ensemble_cmi_score - np.mean(cmi_scores):+.4f})")

print(f"\n✅ 評価完了！結果は {save_dir} に保存されました。")